# Free beamtime on ODIN: starter notebook

No exercise here: **your own question**. Write it down first (below), as in a beamtime proposal at ESS: what do you
want to learn, on which sample, and why neutrons and ODIN can answer it. Then plan your measurements (which
references does your analysis need? how long? in which configuration?), measure in NICOS (button *Open NICOS*), and
analyze your files here. Your own samples can be designed with the *Sample builder*: they appear on the sample changer
of NICOS.

**Your proposal**: choose a proposal number (e.g. your group number, 1000-9999) and enter it in the *Experiment* panel
of NICOS; write it below in `PROPOSAL`. Your files are then in `data/odin/<year>/<proposal>/raw`.

**Tools**
- `rows`: your logbook (one line per count: file, title, settings);
- `red.histograms(file, LAM, ROI)`: image and spectrum of one file, λ from the ESS reduction;
- `red.reduce(sample, open_beam, dark=..., lam=LAM, roi=ROI)`: transmission image and spectrum (official ESS code);
- `curve_fit` (scipy) for fits, `erfc` for smoothed steps; `h5py` to look inside a NeXus file.

## My question

(what, on which sample, why; what result would answer it)

## My plan

(measurements, references, configuration, counting times, and why)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))
import nb_tools

## 1. Your sample

The sample changer of ODIN holds the samples of the library (in NICOS, `sample_info()` describes the one mounted). For your own question, design a sample with the sample builder: blocks, cylinders and spheres of the materials you choose, as many as you want. It is saved with your proposal and appears on the sample changer of NICOS: `move(sample_holder, 'its_name')`.

In [ ]:
nb_tools.sample_builder("odin", "1000")

## 2. Your measurements, in NICOS

Tab *Experiment*: give your *Proposal ID* (the number of PROPOSAL below), then **Apply**. Record with `with nexusfile_open('my title'): count(t=60)`; your files go to the folder of your proposal (below). The measurements and their counting times are your decisions: no budget.

In [ ]:
nb_tools.nicos("odin")

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))     # the modules of the ODIN twin (folder odin/)

In [ ]:
from odin_paths import TWIN_RUN_ROOT
import datetime
PROPOSAL = "1000"          # your proposal number (the one entered in NICOS)
FOLDER = f"{TWIN_RUN_ROOT}/data/odin/{datetime.date.today().year}/{PROPOSAL}/raw"

In [ ]:
import sys, os, json, glob, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath("../.."))
import numpy as np
import scipp as sc
import h5py
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.special import erfc
from odin_paths import TWIN_RUN_ROOT
import odin_reduction as red
import nb_tools
nb_tools.need_measurements(FOLDER, "odin")   # stops here while there is no file

In [ ]:
rows = []
with open(f"{FOLDER}/log.jsonl", encoding="utf-8") as f:
    for line in f:
        entry = json.loads(line)
        if entry["action"] != "count" or not entry["details"].get("file"):
            continue
        d, s = entry["details"], entry["details"]["state"]
        kind = "dark" if s["shutter"] == "closed" else ("open beam" if abs(s["stage_x"]) > 0.02 else "sample")
        rows.append(dict(file=os.path.basename(d["file"]), kind=kind, title=d["title"], t_s=d["t"],
                         counts=d["n_counts"], choppers=s["choppers"], wfm_delta_m=s["wfm_delta"],
                         pinhole_m=s["pinhole"], stage_x_m=s["stage_x"], stage_rot_deg=s["stage_rot"]))
def show_table(rows):                                  # a list of dicts, printed as a table
    if not rows:
        print("no measurement in the logbook"); return
    cols = list(rows[0])
    width = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in cols}
    print("  ".join(c.ljust(width[c]) for c in cols))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(width[c]) for c in cols))

show_table(rows)

## Your analysis

## My answer

(the answer to your question, with uncertainties; what you would do next)

## Submit your work

When you are done: save this notebook (*Save*), then submit. The file for your teacher is made from your measurements and your notebook.

In [ ]:
import submit
_ = submit.Submit("odin/exercises/free_beamtime", "free", title="Free beamtime on ODIN")